## Llama chats with Blender

This notebook shows how Llama chats with Blender through an EMISSOR client. The EMISSOR layer will capture the interaction as a scenario for further analysis. 

https://github.com/ollama/ollama

### Loading Llama

In [3]:
from langchain_ollama import ChatOllama
llm_model = "llama3.2:1b"
#llm_model= "qwen3:0.6b"

llm = ChatOllama(
    model = llm_model,
    temperature = 0.8,
    num_predict = 256,
    # other params ...
)

In [9]:
instruct = { 'role': 'system', 'content': "You are a docter and you will receive questions from patients. Be brief and no more than one sentence and 15 words."}

### Loading Blender

In [5]:
from transformers import BlenderbotTokenizer, BlenderbotForConditionalGeneration
mname = 'facebook/blenderbot-400M-distill'
blender_model = BlenderbotForConditionalGeneration.from_pretrained(mname)
blender_tokenizer = BlenderbotTokenizer.from_pretrained(mname)

In [6]:
context_size = 5
def get_answer_from_blender(prompt:str, history_list:[]):
    answer = ""
    sentences = []
    history = ""
    for i, his in enumerate(history):
        if i==context_size:
            break
        history += his['content'] +". "
    input_prompt = history+prompt
    bot_input_ids = blender_tokenizer(input_prompt, return_tensors='pt')
    chat_history_ids = blender_model.generate(**bot_input_ids)
    utteranceList = blender_tokenizer.batch_decode(chat_history_ids)
    answer = utteranceList[0].strip('</s>')
    return answer

### Creating an EMISSOR client

In [10]:
from leolani_client import LeolaniChatClient
emissor_path = "./emissor"
HUMAN="BlenderBot"
AGENT="LLM"
leolaniClient = LeolaniChatClient(emissor_path=emissor_path, agent=AGENT, human=HUMAN)

### Interaction loop

In [11]:
history = []
history.append(instruct)
print(history)
### First prompt
response = llm.invoke(history)
utterance = response.content
print(AGENT + ": " + utterance)
leolaniClient._add_utterance(AGENT, utterance) 
prompt = { 'role': 'system', 'content': utterance}
history.append(prompt)

utterance = get_answer_from_blender(utterance, history)
print('\n\t'+HUMAN + ": " + utterance)
prompt = { 'role': 'user', 'content': utterance}
history.append(prompt)
leolaniClient._add_utterance(AGENT, prompt)

max_count = 5
counter = 0

while counter < max_count:
    counter +=1
    # Create the response from the system and store this as a new signal
    response = llm.invoke(history)
    utterance = response.content
    print(AGENT + ": " + utterance)
    leolaniClient._add_utterance(AGENT, utterance) 
    prompt = { 'role': 'system', 'content': utterance}
    history.append(prompt)

    utterance = get_answer_from_blender(utterance, history)
    print('\n\t'+HUMAN + ": " + utterance)
    prompt = { 'role': 'user', 'content': utterance}
    history.append(prompt)
    leolaniClient._add_utterance(AGENT, prompt)

##### After completion, we save the scenario in the defined emissor folder.
leolaniClient._save_scenario() 

[{'role': 'system', 'content': 'You are a docter and you will receive questions from patients. Be brief and no more than one sentence and 15 words.'}]
LLM: <|start_header_id|>assistant<|end_header_id|>

I'm ready to help. Go ahead with your question.

	BlenderBot:  Thank you so much, I appreciate it so much. I've been trying to figure out what to do.
LLM: Please describe the situation or symptom you're experiencing to get a better understanding of your concern.

	BlenderBot:  I'm not sure what I'm going to do. I don't know what I should do.
LLM: Feelings of uncertainty are common when faced with a difficult decision.

What are your biggest concerns about each option?

	BlenderBot:  My biggest concern is that I am not sure what I want to do with my life. 
LLM: It can be really tough when you're feeling uncertain about your future.

Have you considered talking to a therapist or counselor who can help you explore your feelings and find guidance?

	BlenderBot:  I have, but I don't think it